# 26 — Validação da saída coerente para o usuário

## TL;DR

A saída principal agora segue uma única narrativa: alegação do artigo, alcance da leitura, balanço das evidências, interpretação, trechos rastreáveis, limitações e próximo passo. Os percentuais técnicos continuam disponíveis, mas ficam recolhidos em detalhes técnicos.

## Contexto e método

Este notebook é um teste determinístico de apresentação. Ele usa três cenários controlados — compatibilidade predominante, divergência e ausência de comparação direta — para verificar linguagem, denominadores, abstenção e proveniência. Não avalia precisão clínica.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from fatofake.result_presentation import build_user_summary

## Dados

Cada artigo abaixo possui apenas os campos necessários para construir a apresentação. O primeiro preserva uma citação localizada em Resultados, página 6.

In [2]:
def article(pmid, relation, access='FULL_TEXT', page=None):
    return {
        'pmid': pmid, 'title': f'Estudo {pmid}', 'publication_date': '2025',
        'url': f'https://example.org/{pmid}', 'access_level': access,
        'quality': {'study_design': 'OBSERVATIONAL'},
        'assessments': [{
            'relation': relation, 'confidence': 0.84,
            'evidence': {
                'text': f'Resultado rastreável do estudo {pmid}.',
                'section': 'Resultados', 'page': page,
                'content_scope': access, 'source_url': f'https://example.org/{pmid}',
            },
        }],
    }

base = {
    'submitted_article': {
        'primary_claim': 'A intervenção reduz sintomas em adultos.',
        'content_scope': 'ABSTRACT_ONLY',
    }
}
scenarios = {
    'compatível': {**base, 'articles': [article('A', 'SUPPORTS', page=6), article('B', 'NEUTRAL', 'ABSTRACT_ONLY')]},
    'divergente': {**base, 'articles': [article('A', 'SUPPORTS'), article('B', 'CONTRADICTS')]},
    'sem comparação': {**base, 'articles': [{'pmid': 'A', 'title': 'Somente metadados', 'access_level': 'METADATA_ONLY', 'assessments': []}]},
}

## Resultados

In [3]:
summaries = {name: build_user_summary(payload) for name, payload in scenarios.items()}
for name, summary in summaries.items():
    print(f'\n[{name.upper()}]')
    print(summary['headline'])
    print(summary['summary'])
    print('Como interpretar:', summary['interpretation'])
    print('Leitura:', summary['reading']['summary'])
    print('Próximo passo:', summary['next_action'])


[COMPATÍVEL]
Há compatibilidade preliminar com a literatura recuperada
Foram localizados 2 artigos; 2 tiveram trechos comparados com a alegação. A comparação encontrou 1 compatível, 0 divergentes, 1 apenas contextual e 0 inconclusivos.
Como interpretar: Os trechos localizados caminham na mesma direção da alegação, mas isso não comprova que o artigo esteja correto nem elimina limitações metodológicas. A direção observada depende de apenas um artigo comparável. Parte da leitura foi limitada ao abstract.
Leitura: Artigo enviado: somente abstract. Foram recuperados 2 artigos independentes; 2 foram comparados: 1 com texto completo e 1 somente pelo abstract.
Próximo passo: Envie o PDF do artigo para analisar métodos, resultados, tabelas e páginas; depois confira manualmente os trechos independentes abaixo.

[DIVERGENTE]
Os estudos recuperados apresentam resultados divergentes
Foram localizados 2 artigos; 2 tiveram trechos comparados com a alegação. A comparação encontrou 1 compatível, 1 div

In [4]:
compatible = summaries['compatível']
assert compatible['status'] == 'PREDOMINANTLY_COMPATIBLE'
assert compatible['summary'].startswith('Foram localizados 2 artigos; 2 tiveram')
assert 'compatível(is)' not in compatible['summary']
assert compatible['findings'][0]['location'] == 'Seção Resultados, página 6'
assert summaries['divergente']['status'] == 'MIXED'
assert summaries['sem comparação']['status'] == 'NO_DIRECT_COMPARISON'
assert summaries['sem comparação']['findings'] == []
'Todos os cenários preservaram coerência, abstenção e proveniência.'

'Todos os cenários preservaram coerência, abstenção e proveniência.'

## Conclusões

A camada de apresentação elimina códigos internos e percentuais isolados da leitura principal. A pessoa vê primeiro o que foi comparado e o significado do resultado; depois pode inspecionar cada citação e, opcionalmente, abrir métricas técnicas. O cenário sem evidência direta se abstém explicitamente, enquanto o cenário divergente não rotula o artigo como falso.